# 03b — Panel con efectos fijos de dos vías

**Proyecto:** Análisis de Elasticidad de Demanda en Delivery de Comida (v2)
**Fase:** 3b — Robustez con panel y clustering

Este notebook responde a una limitación explícita de la Fase 3 (ver bitácora,
sección 4.4): el modelo OLS con controles no incluía efectos fijos por
semana ni por la combinación específica centro-comida, y sus errores
estándar no consideraban la posible dependencia dentro de cada centro.

Objetivos:
1. Estimar un modelo de panel con efectos fijos de **dos vías** (entidad =
   combinación centro-comida; tiempo = semana), con errores clusterizados
2. Comparar la elasticidad resultante contra la de la Fase 3 (-0.759)
3. Extra 1: bootstrap clusterizado (por centro) como robustez adicional
4. Extra 2: test de Shapiro-Wilk submuestreado sobre los residuos


In [1]:
import sys
sys.path.append('../src')

import json
from pathlib import Path

import pandas as pd
import numpy as np

from modeling import (
    modelo_panel_efectos_fijos,
    bootstrap_elasticidad_clusterizado,
)
from diagnostics import shapiro_wilk_submuestreado

pd.set_option('display.max_columns', None)

## 1. Cargar datos

In [2]:
df = pd.read_csv('../data/processed/demanda_limpia.csv')
print(f'Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}')
print(f'Centros únicos: {df["center_id"].nunique()} | Comidas únicas: {df["meal_id"].nunique()}')
print(f'Combinaciones centro-comida posibles: {df["center_id"].nunique() * df["meal_id"].nunique()}')
print(f'Combinaciones centro-comida presentes: {df.groupby(["center_id", "meal_id"]).ngroups}')

Filas: 456,548 | Columnas: 20
Centros únicos: 77 | Comidas únicas: 51
Combinaciones centro-comida posibles: 3927
Combinaciones centro-comida presentes: 3597


## 2. Modelo de panel con efectos fijos de dos vías

**Especificación:** `log_num_orders ~ log_checkout_price + emailer_for_promotion + homepage_featured + EntityEffects + TimeEffects`

- **EntityEffects**: un efecto fijo por cada combinación centro-comida (controla heterogeneidad no observada específica de ese par: por ejemplo, si cierto centro es sistemáticamente mejor vendiendo cierta comida por razones no capturadas en los datos)
- **TimeEffects**: un efecto fijo por semana (controla tendencias/estacionalidad agregada que afecta a todas las combinaciones por igual esa semana)
- **cov_type='clustered', cluster_entity=True**: errores estándar que permiten correlación arbitraria entre observaciones de la misma entidad a lo largo del tiempo

**Nota técnica:** se usa `linearmodels.PanelOLS`, que implementa estos efectos fijos mediante la transformación *within* (demeaning), no mediante variables dummy explícitas — con miles de combinaciones centro-comida posibles, una matriz de dummies sería inviable en memoria.

In [3]:
resultado_panel = modelo_panel_efectos_fijos(df)
print(resultado_panel)

                          PanelOLS Estimation Summary                           
Dep. Variable:         log_num_orders   R-squared:                        0.3208
Estimator:                   PanelOLS   R-squared (Between):             -7.1273
No. Observations:              456548   R-squared (Within):               0.3215
Date:                Thu, Oct 22 2026   R-squared (Overall):             -6.7011
Time:                        02:22:32   Log-likelihood                -3.446e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                   7.129e+04
Entities:                        3597   P-value                           0.0000
Avg Obs:                       126.92   Distribution:                F(3,452804)
Min Obs:                       1.0000                                           
Max Obs:                       145.00   F-statistic (robust):             9508.1
                            

## 3. Comparación con el modelo de la Fase 3

| Modelo | Elasticidad |
|---|---|
| OLS con controles (Fase 3) | -0.759 |
| Panel con efectos fijos de dos vías (este notebook) | -1.5985 (EE agrupado 0.0239) |

**Qué esperamos:** si la elasticidad se mantiene en un rango similar, es evidencia de que el resultado de la Fase 3 era robusto pese a no controlar explícitamente por semana/combinación centro-comida. Si cambia sustancialmente, indica que esos efectos no observados eran relevantes.

In [4]:
from modeling import modelo_log_log_controles
elasticidad_ols = modelo_log_log_controles(df).params['log_checkout_price']
elasticidad_panel = resultado_panel.params['log_checkout_price']
ee_panel = resultado_panel.std_errors['log_checkout_price']
print(f'Elasticidad (panel FE dos vías): {elasticidad_panel:.4f}')
print(f'Error estándar clusterizado: {ee_panel:.4f}')
print(f'Elasticidad OLS con controles (Fase 3): {elasticidad_ols:.4f}')
print(f'Diferencia: {abs(elasticidad_panel - elasticidad_ols):.4f}')

Elasticidad (panel FE dos vías): -1.5985
Error estándar clusterizado: 0.0239
Elasticidad OLS con controles (Fase 3): -0.7590
Diferencia: 0.8396


## 4. Extra 1 — Bootstrap por centro (resultado precalculado)

El pairs bootstrap por centro (1000 réplicas, seed=42, remuestreo de los 77 centros con
reetiquetado de entidades) tarda horas, por lo que se ejecuta con
`scripts/analysis/vm_pairs_bootstrap_panel_fe.py`. Aquí se carga su resumen.

In [5]:
boot = np.load('../reports/diagnostics/pairs_bootstrap_panel_fe_results.npy', allow_pickle=True).item()
for k in ['coef_base', 'coef_media', 'coef_std', 'ic_95_inferior', 'ic_95_superior', 'n_iteraciones_validas']:
    print(f'{k:>22}: {boot[k]}')

             coef_base: -1.5985291480156154
            coef_media: -1.5991985845085437
              coef_std: 0.027130459650650467
        ic_95_inferior: -1.650603207205144
        ic_95_superior: -1.5455251046081935
 n_iteraciones_validas: 1000


## 5. Extra 2 — Shapiro-Wilk submuestreado

Complementa el test de Jarque-Bera de la Fase 4 (que ya se hizo sobre el
modelo OLS). Shapiro-Wilk es más potente para detectar desviaciones de
normalidad en muestras chicas, pero no está diseñado para ~450,000
observaciones — se evalúa sobre una submuestra aleatoria, documentando
explícitamente esta limitación (no es una prueba sobre el dataset completo).

In [6]:
resultado_shapiro = shapiro_wilk_submuestreado(resultado_panel, n=5000)
print(resultado_shapiro)

{'n_submuestra': 5000, 'shapiro_estadistico': np.float64(0.9829139619811963), 'shapiro_pvalor': np.float64(4.040947215498977e-24), 'nota': 'Evaluado sobre una submuestra aleatoria de 5000 de 456548 residuos totales, no sobre el dataset completo.'}


## 6. Guardar resultados de la Fase 3b

## Próximos pasos (Fase 3c — variables instrumentales)
- Construir instrumentos: `base_price` y precio promedio en otros centros (leave-one-out)
- Estimar 2SLS
- Test de instrumento débil (F de primera etapa) y sobre-identificación (Hansen J)